In [497]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [498]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import joblib
import pandas as pd

from src.data import load_matches
from src.elo import prepare_matches, update_elo
from src.simulation import predict_match, simulate_match, simulate_group, simulate_group_stage, simulate_knockout_match, construct_round32, simulate_knockout_round, construct_round16, construct_QF, construct_SF
from src.features import update_team_history, update_h2h, build_features, assign_third_place_slots
import numpy as np
from collections import Counter
from src.simulation import run_tournament

#Load everything from earlier notebooks

history_dict = joblib.load('final_history.joblib')
h2h_dict = joblib.load('final_h2h.joblib')
country_elo = joblib.load('final_elo.joblib')
model_h = joblib.load('home_goals_model.joblib')
model_a = joblib.load('away_goals_model.joblib')
features = joblib.load('model_features.joblib')

#Make a dataframe containing all the games in the group stage
group_stage_matches = load_matches()
group_stage_matches = prepare_matches(group_stage_matches, "2026-06-10", "2026-06-28")
df_confederations = pd.read_csv('../data/FIFA_confederations.csv')

wc_groups = {
    "A": ["Mexico", "South Korea", "South Africa", "Czech Republic"],
    "B": ["Canada", "Switzerland", "Qatar", "Bosnia and Herzegovina"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

In [499]:
team_to_group = {team: group for group, teams in wc_groups.items() for team in teams}
team_to_confederation = dict(zip(df_confederations["nation"], df_confederations["confederation"]))

# Create group_stages equivalent
rows = []
for group, teams in wc_groups.items():
    for i, team in enumerate(teams, 1):
        rows.append({"group": group, "position": i, "nation": team})
df_groups = pd.DataFrame(rows)

# Create fixtures equivalent — you already have this
df_group_fixtures = group_stage_matches.copy()
df_group_fixtures["group"] = df_group_fixtures["home_team"].map(team_to_group)

In [500]:
#Example run
x=build_features('Argentina', 'Brazil', country_elo, team_to_confederation, features)
x


,const,home_elo,away_elo,neutral,tournament_weight,home_confederation_AFC,home_confederation_CAF,home_confederation_CONCACAF,home_confederation_CONMEBOL,home_confederation_OFC,home_confederation_UEFA,home_confederation_Unknown,away_confederation_AFC,away_confederation_CAF,away_confederation_CONCACAF,away_confederation_CONMEBOL,away_confederation_OFC,away_confederation_UEFA,away_confederation_Unknown
0,1.0,2113.427162,1989.655735,1.0,5.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [501]:
y = simulate_match('Argentina', 'Brazil', model_h, model_a, country_elo, team_to_confederation, features)
y

{'home_team': 'Argentina',
 'away_team': 'Brazil',
 'home_goals': 1,
 'away_goals': 0,
 'result': 'home_win',
 'winner': 'Argentina'}

In [502]:
group_a_table, group_a_matches = simulate_group("C", df_groups, df_group_fixtures, model_h, model_a, country_elo, team_to_confederation, features)

display(group_a_matches)
display(group_a_table)

,home_team,away_team,home_goals,away_goals,result,winner
0,Brazil,Morocco,0,0,draw,NaN
1,Haiti,Scotland,2,2,draw,NaN
2,Scotland,Morocco,1,2,away_win,Morocco
3,Brazil,Haiti,2,0,home_win,Brazil
4,Scotland,Brazil,1,2,away_win,Brazil
5,Morocco,Haiti,2,3,away_win,Haiti


,team,played,wins,draws,losses,goals_for,goals_against,goal_difference,points,group_rank,group
0,Brazil,3,2,1,0,4,1,3,7,1,C
1,Morocco,3,1,1,1,4,4,0,4,2,C
2,Haiti,3,1,1,1,5,6,-1,4,3,C
3,Scotland,3,0,1,2,4,6,-2,1,4,C


In [503]:
# group_c_rank_results = []

# for i in range(1000):
#     group_table, group_matches = simulate_group("C", df_groups, df_group_fixtures, model_h, model_a, country_elo, team_to_confederation, features)

#     group_c_rank_results.append(
#         group_table[["team", "group_rank"]]
#     )

# df_group_c_ranks = pd.concat(group_c_rank_results, ignore_index=True)

# group_c_rank_probs = (
#     df_group_c_ranks
#     .groupby(["team", "group_rank"])
#     .size()
#     .reset_index(name="count")
# )

# group_c_rank_probs["probability"] = group_c_rank_probs["count"] / 1000

# group_c_rank_probs_pivot = (
#     group_c_rank_probs
#     .pivot(index="team", columns="group_rank", values="probability")
#     .fillna(0)
# )

# group_c_rank_probs_pivot.columns = [
#     f"rank_{int(col)}_prob" for col in group_c_rank_probs_pivot.columns
# ]

# group_c_rank_probs_pivot.sort_values("rank_1_prob", ascending=False)

In [504]:
group_stage = simulate_group_stage(df_groups, df_group_fixtures, model_h, model_a, country_elo, team_to_confederation, features)
group_stage_result = group_stage[0]

top2 = group_stage_result.groupby('group').head(2).copy() #Teams that placed top 2 in their group which qualifies
third = group_stage_result.groupby('group').nth(2).copy() #All teams that placed third (only 8 of them move on)
best8_third = third.sort_values(
    ['points', 'goal_difference', 'goals_for'], 
    ascending=[False, False, False]
).head(8).reset_index(drop=True)

best8_third['third_place_rank'] = best8_third.index + 1

display(group_stage_result)

,team,played,wins,draws,losses,goals_for,goals_against,goal_difference,points,group_rank,group
0,South Korea,3,2,1,0,4,2,2,7,1,A
1,Mexico,3,1,1,1,4,2,2,4,2,A
2,South Africa,3,1,1,1,2,4,-2,4,3,A
3,Czech Republic,3,0,1,2,2,4,-2,1,4,A
4,Switzerland,3,2,1,0,8,1,7,7,1,B
5,Canada,3,2,0,1,8,5,3,6,2,B
6,Qatar,3,1,0,2,3,9,-6,3,3,B
7,Bosnia and Herzegovina,3,0,1,2,3,7,-4,1,4,B
8,Brazil,3,3,0,0,4,0,4,9,1,C
9,Morocco,3,1,1,1,3,2,1,4,2,C


In [505]:
round_of_32 = pd.concat([top2, best8_third])
round_of_32

,team,played,wins,draws,losses,goals_for,goals_against,goal_difference,points,group_rank,group,third_place_rank
0,South Korea,3,2,1,0,4,2,2,7,1,A,NaN
1,Mexico,3,1,1,1,4,2,2,4,2,A,NaN
4,Switzerland,3,2,1,0,8,1,7,7,1,B,NaN
5,Canada,3,2,0,1,8,5,3,6,2,B,NaN
8,Brazil,3,3,0,0,4,0,4,9,1,C,NaN
9,Morocco,3,1,1,1,3,2,1,4,2,C,NaN
12,Turkey,3,1,2,0,4,3,1,5,1,D,NaN
13,Paraguay,3,1,2,0,1,0,1,5,2,D,NaN
16,Ecuador,3,2,1,0,6,1,5,7,1,E,NaN
17,Germany,3,2,0,1,6,3,3,6,2,E,NaN


In [506]:
third_place_team_to_group = dict(
    zip(best8_third["team"], best8_third["group"])
)

third_place_group_to_team = dict(
    zip(best8_third["group"], best8_third["team"])
)

print("Qualified third-place teams by group:")
third_place_group_to_team

Qualified third-place teams by group:


{'H': 'Cape Verde',
 'G': 'Belgium',
 'K': 'Portugal',
 'A': 'South Africa',
 'D': 'Australia',
 'F': 'Sweden',
 'I': 'Norway',
 'J': 'Algeria'}

In [507]:
rd32_teams = {}
for team in top2.itertuples(index=False):
        print(f"{team.group_rank}{team.group}, {team.team}")
        rd32_teams[f'{team.group_rank}{team.group}'] = team.team

print("\nBest third-placed teams:")
display(best8_third[[
    "team",
    "group",
    "points",
    "goal_difference",
    "goals_for",
    "third_place_rank"
]])

1A, South Korea
2A, Mexico
1B, Switzerland
2B, Canada
1C, Brazil
2C, Morocco
1D, Turkey
2D, Paraguay
1E, Ecuador
2E, Germany
1F, Netherlands
2F, Japan
1G, Iran
2G, Egypt
1H, Spain
2H, Uruguay
1I, France
2I, Senegal
1J, Jordan
2J, Argentina
1K, Colombia
2K, Uzbekistan
1L, Croatia
2L, England

Best third-placed teams:


,team,group,points,goal_difference,goals_for,third_place_rank
0,Cape Verde,H,4,0,5,1
1,Belgium,G,4,0,3,2
2,Portugal,K,4,0,2,3
3,South Africa,A,4,-2,2,4
4,Australia,D,3,0,2,5
5,Sweden,F,3,-1,2,6
6,Norway,I,3,-1,2,7
7,Algeria,J,3,-2,2,8


In [508]:
thirds = assign_third_place_slots(best8_third)
rd32_teams |= thirds

x = construct_round32(rd32_teams)
print(rd32_teams)
simulate_knockout_match(x[73][0], x[73][1], model_h, model_a, country_elo, team_to_confederation, features)

{'1A': 'South Korea', '2A': 'Mexico', '1B': 'Switzerland', '2B': 'Canada', '1C': 'Brazil', '2C': 'Morocco', '1D': 'Turkey', '2D': 'Paraguay', '1E': 'Ecuador', '2E': 'Germany', '1F': 'Netherlands', '2F': 'Japan', '1G': 'Iran', '2G': 'Egypt', '1H': 'Spain', '2H': 'Uruguay', '1I': 'France', '2I': 'Senegal', '1J': 'Jordan', '2J': 'Argentina', '1K': 'Colombia', '2K': 'Uzbekistan', '1L': 'Croatia', '2L': 'England', '3ABCDF': 'South Africa', '3CDFGH': 'Cape Verde', '3CEFHI': 'Sweden', '3EHIJK': 'Portugal', '3BEFIJ': 'Norway', '3AEHIJ': 'Algeria', '3EFGIJ': 'Belgium', '3DEIJL': 'Australia'}


{'home_team': 'Mexico',
 'away_team': 'Canada',
 'home_goals': 1,
 'away_goals': 2,
 'result': 'away_win',
 'result_type': 'regular_time',
 'winner': 'Canada',
 'loser': 'Mexico',
 'home_win_prob': np.float64(0.5152247527538497),
 'draw_prob': np.float64(0.25678157589795825),
 'away_win_prob': np.float64(0.22795931322280022)}

In [509]:
winners, results = simulate_knockout_round(x, model_h, model_a, country_elo, team_to_confederation, features)

display(results)

,home_team,away_team,home_goals,away_goals,result_type,winner,loser,home_win_prob,draw_prob,away_win_prob
0,Mexico,Canada,0,0,OT/Pens,Canada,Mexico,0.515225,0.256782,0.227959
1,Ecuador,South Africa,4,0,regular_time,Ecuador,South Africa,0.822352,0.131444,0.045224
2,Netherlands,Morocco,1,1,OT/Pens,Morocco,Netherlands,0.357884,0.304442,0.337672
3,Brazil,Japan,0,0,OT/Pens,Brazil,Japan,0.422391,0.304137,0.273468
4,France,Cape Verde,0,1,regular_time,Cape Verde,France,0.792673,0.149488,0.057197
5,Ecuador,Senegal,0,0,OT/Pens,Ecuador,Senegal,0.537763,0.276309,0.185912
6,South Korea,Sweden,1,1,OT/Pens,South Korea,Sweden,0.490674,0.271166,0.238142
7,Croatia,Portugal,2,1,regular_time,Croatia,Portugal,0.233953,0.271413,0.494616
8,Turkey,Norway,0,0,OT/Pens,Norway,Turkey,0.294086,0.283923,0.421983
9,Iran,Algeria,2,2,OT/Pens,Iran,Algeria,0.437502,0.295970,0.266522


In [510]:
r16_matchups = construct_round16(winners)

winners, results = simulate_knockout_round(r16_matchups, model_h, model_a, country_elo, team_to_confederation, features)

display(results)

,home_team,away_team,home_goals,away_goals,result_type,winner,loser,home_win_prob,draw_prob,away_win_prob
0,Ecuador,Cape Verde,1,0,regular_time,Ecuador,Cape Verde,0.772489,0.163046,0.064029
1,Canada,Morocco,1,0,regular_time,Canada,Morocco,0.362721,0.302135,0.335141
2,Brazil,Ecuador,5,0,regular_time,Brazil,Ecuador,0.419022,0.292020,0.288952
3,South Korea,Croatia,0,1,regular_time,Croatia,South Korea,0.275683,0.282806,0.441501
4,England,Argentina,0,0,OT/Pens,Argentina,England,0.181786,0.252760,0.565414
5,Norway,Iran,2,0,regular_time,Norway,Iran,0.393744,0.295963,0.310288
6,Uruguay,Paraguay,1,1,OT/Pens,Uruguay,Paraguay,0.431630,0.286201,0.282162
7,Switzerland,Colombia,0,2,regular_time,Colombia,Switzerland,0.194713,0.253012,0.552234


In [511]:
QF_matchsups = construct_QF(winners)

winners, results = simulate_knockout_round(QF_matchsups, model_h, model_a, country_elo, team_to_confederation, features)

display(results)

,home_team,away_team,home_goals,away_goals,result_type,winner,loser,home_win_prob,draw_prob,away_win_prob
0,Ecuador,Canada,0,0,OT/Pens,Ecuador,Canada,0.517306,0.261464,0.221202
1,Argentina,Norway,0,0,OT/Pens,Argentina,Norway,0.610885,0.243036,0.146026
2,Brazil,Croatia,1,1,OT/Pens,Brazil,Croatia,0.492045,0.279971,0.227972
3,Uruguay,Colombia,1,2,regular_time,Colombia,Uruguay,0.282329,0.288022,0.429642


In [512]:
SF_matchsups = construct_SF(winners)

winners, results = simulate_knockout_round(SF_matchsups, model_h, model_a, country_elo, team_to_confederation, features)

display(results)

,home_team,away_team,home_goals,away_goals,result_type,winner,loser,home_win_prob,draw_prob,away_win_prob
0,Ecuador,Argentina,0,3,regular_time,Argentina,Ecuador,0.199843,0.266995,0.533139
1,Brazil,Colombia,3,0,regular_time,Brazil,Colombia,0.375497,0.296428,0.328071


In [513]:
winner, results = simulate_knockout_round({103: (winners[101], winners[102])}, model_h, model_a, country_elo, team_to_confederation, features)

display(results)

print(f'{winner[103]} won the World Cup')

,home_team,away_team,home_goals,away_goals,result_type,winner,loser,home_win_prob,draw_prob,away_win_prob
0,Argentina,Brazil,3,3,OT/Pens,Brazil,Argentina,0.487406,0.283162,0.229421


Brazil won the World Cup
